# SSMs and hybrid models

How structured state space models work, why they complement attention,
and how to build custom hybrids with `block_configs`.

In [ ]:
import mlx.core as mx

from lmxlab.analysis.activations import ActivationCapture
from lmxlab.core.config import BlockConfig, ModelConfig
from lmxlab.models.base import LanguageModel
from lmxlab.models.falcon import falcon_h1_tiny

## SSM recap

An SSM (Gu et al., 2021) maintains a hidden state that evolves recurrently:

$$\mathbf{h}_t = \mathbf{A}\,\mathbf{h}_{t-1} + \mathbf{B}\,x_t$$
$$y_t = \mathbf{C}\,\mathbf{h}_t + D\,x_t$$

This gives $O(n)$ complexity and fixed memory, but the state is a lossy
compression — the model can't look up an arbitrary past token the way attention can.

Mamba (Gu & Dao, 2023) makes $\mathbf{B}$, $\mathbf{C}$, and $\Delta$ input-dependent
("selective"), so the model learns *what* to keep in state.
Mamba-2 (Dao & Gu, 2024) reformulates this through a duality with attention
for efficient chunked computation on hardware.

## Inspecting a hybrid

Falcon-H1 uses an MMM* pattern — 3 Mamba-2 blocks per 1 GQA block.
The SSM layers handle sequential patterns cheaply; the attention layers
provide exact token recall when needed.

In [ ]:
cfg = falcon_h1_tiny()
print(f"{cfg.n_layers} layers:\n")

for i in range(cfg.n_layers):
    bcfg = cfg.get_block_config(i)
    if "mamba" in bcfg.attention:
        print(
            f"  layer {i}: Mamba-2 "
            f"(heads={bcfg.mamba_n_heads}, "
            f"head_dim={bcfg.mamba_head_dim}, "
            f"state={bcfg.ssm_state_size})"
        )
    else:
        n_kv = bcfg.effective_n_kv_heads
        print(f"  layer {i}: GQA (heads={bcfg.n_heads}, kv_heads={n_kv})")

In [ ]:
model = LanguageModel(cfg)
mx.eval(model.parameters())

tokens = mx.zeros((1, 32), dtype=mx.int32)
logits, caches = model(tokens)
mx.eval(logits)
print(f"{model.count_parameters():,} params, output {logits.shape}")

## Building a custom hybrid

`ModelConfig` takes a `block_configs` tuple to assign different configs per layer.
Here we make a 6-layer model with pattern MMMAMM (mostly SSM, one attention layer in the middle).

In [ ]:
attn_block = BlockConfig(
    attention="gqa",
    ffn="gated",
    norm="rms_norm",
    position="rope",
    d_model=64,
    n_heads=4,
    n_kv_heads=2,
    d_ff=128,
    bias=False,
    max_seq_len=128,
)
ssm_block = BlockConfig(
    attention="mamba2",
    ffn="gated",
    norm="rms_norm",
    position="none",
    d_model=64,
    n_heads=4,
    n_kv_heads=2,
    d_ff=128,
    bias=False,
    max_seq_len=128,
    mamba_n_heads=4,
    mamba_head_dim=32,
    ssm_state_size=64,
    mamba_expand=2,
)

pattern = "MMMAMM"
blocks = tuple(attn_block if c == "A" else ssm_block for c in pattern)

hybrid = LanguageModel(
    ModelConfig(
        block=attn_block,
        vocab_size=256,
        n_layers=len(pattern),
        block_configs=blocks,
        tie_embeddings=True,
    )
)
mx.eval(hybrid.parameters())

out, _ = hybrid(mx.zeros((1, 16), dtype=mx.int32))
mx.eval(out)
print(f"{hybrid.count_parameters():,} params, output {out.shape}")

## Activation norms through the stack

How do SSM vs attention layers transform the residual stream?

In [ ]:
with ActivationCapture(hybrid) as cap:
    hybrid(mx.zeros((1, 16), dtype=mx.int32))

norms = cap.layer_norms()
print(f"{'layer':>5} {'type':>5} {'in':>8} {'out':>8}")
print("-" * 28)
for i, c in enumerate(pattern):
    ltype = "Attn" if c == "A" else "SSM"
    in_n = norms.get(f"layer_{i}/input", 0)
    out_n = norms.get(f"layer_{i}/output", 0)
    print(f"{i:>5} {ltype:>5} {in_n:>8.3f} {out_n:>8.3f}")

## Published hybrids

Factory functions for Falcon-H1, Jamba (Lieber et al., 2024), and Bamba.

In [ ]:
from lmxlab.models.jamba import jamba_tiny

for name, factory in [
    ("Falcon-H1", falcon_h1_tiny),
    ("Jamba", jamba_tiny),
]:
    c = factory()
    m = LanguageModel(c)
    mx.eval(m.parameters())
    n_ssm = sum(
        1
        for i in range(c.n_layers)
        if "mamba" in c.get_block_config(i).attention
    )
    n_attn = c.n_layers - n_ssm
    print(
        f"{name}: {m.count_parameters():,} params, "
        f"{n_ssm} SSM + {n_attn} attn layers"
    )